In [1]:
pip install pyspark

In [2]:
import os
os.environ["JAVA_HOME"]="/lib/jvm/java-11-openjdk-amd64"

In [3]:
!apt-get install openjdk-11-jdk-headless -qq
!pip install -q pyspark

In [4]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SQLContext

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("films") \
    .getOrCreate()

sc = spark.sparkContext

In [10]:
house_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/content/Boston.csv")

house_df.show()

+---+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|_c0|   crim|  zn|indus|chas|  nox|   rm|  age|   dis|rad|tax|ptratio| black|lstat|medv|
+---+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|  1|0.00632|18.0| 2.31|   0|0.538|6.575| 65.2|  4.09|  1|296|   15.3| 396.9| 4.98|24.0|
|  2|0.02731| 0.0| 7.07|   0|0.469|6.421| 78.9|4.9671|  2|242|   17.8| 396.9| 9.14|21.6|
|  3|0.02729| 0.0| 7.07|   0|0.469|7.185| 61.1|4.9671|  2|242|   17.8|392.83| 4.03|34.7|
|  4|0.03237| 0.0| 2.18|   0|0.458|6.998| 45.8|6.0622|  3|222|   18.7|394.63| 2.94|33.4|
|  5|0.06905| 0.0| 2.18|   0|0.458|7.147| 54.2|6.0622|  3|222|   18.7| 396.9| 5.33|36.2|
|  6|0.02985| 0.0| 2.18|   0|0.458| 6.43| 58.7|6.0622|  3|222|   18.7|394.12| 5.21|28.7|
|  7|0.08829|12.5| 7.87|   0|0.524|6.012| 66.6|5.5605|  5|311|   15.2| 395.6|12.43|22.9|
|  8|0.14455|12.5| 7.87|   0|0.524|6.172| 96.1|5.9505|  5|311|   15.2| 396.9|19.15|27.1|
|  9|0.21124|12.5| 7.

In [11]:
## Printing schema
house_df.printSchema()


root
 |-- _c0: integer (nullable = true)
 |-- crim: double (nullable = true)
 |-- zn: double (nullable = true)
 |-- indus: double (nullable = true)
 |-- chas: integer (nullable = true)
 |-- nox: double (nullable = true)
 |-- rm: double (nullable = true)
 |-- age: double (nullable = true)
 |-- dis: double (nullable = true)
 |-- rad: integer (nullable = true)
 |-- tax: integer (nullable = true)
 |-- ptratio: double (nullable = true)
 |-- black: double (nullable = true)
 |-- lstat: double (nullable = true)
 |-- medv: double (nullable = true)



In [12]:
## Descriptive analysis
house_df.toPandas()

,_c0,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
0,1,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,2,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,3,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,4,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,5,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,502,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,9.67,22.4
502,503,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,504,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9
504,505,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0


In [13]:
from pyspark.ml.feature import VectorAssembler
vectorAssembler = VectorAssembler(inputCols = ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'black', 'lstat'], outputCol = 'features')
vhouse_df = vectorAssembler.transform(house_df)
vhouse_df = vhouse_df.select(['features', 'medv'])
vhouse_df.show(3)

+--------------------+----+
|            features|medv|
+--------------------+----+
|[0.00632,18.0,2.3...|24.0|
|[0.02731,0.0,7.07...|21.6|
|[0.02729,0.0,7.07...|34.7|
+--------------------+----+
only showing top 3 rows


In [14]:
splits = vhouse_df.randomSplit([0.7, 0.3])
train_df = splits[0]
test_df = splits[1]
#train_df,test_df=vhouse_df.randomSplit([0.7,0.3])

In [15]:
from pyspark.ml.regression import LinearRegression
lr = LinearRegression(featuresCol = 'features', labelCol='medv', maxIter=10)
lr_model = lr.fit(train_df)
print("Coefficients: " + str(lr_model.coefficients))
print("Intercept: " + str(lr_model.intercept))

Coefficients: [-0.1183855299354755,0.051037853755733005,0.03415351994106921,2.005676679598753,-18.662503931582055,3.585966267259749,0.009213285358719896,-1.4711936757528443,0.27642921747845467,-0.010861461383565208,-0.9570310368102878,0.009223996900364066,-0.559979861448899]
Intercept: 37.96396515600329


In [16]:
trainingSummary = lr_model.summary
print("RMSE: %f" % trainingSummary.rootMeanSquaredError)
print("r2: %f" % trainingSummary.r2)

RMSE: 4.844923
r2: 0.724308


In [17]:
lr_predictions = lr_model.transform(test_df)
lr_predictions.select("prediction","medv","features").show(5)
from pyspark.ml.evaluation import RegressionEvaluator
lr_evaluator = RegressionEvaluator(predictionCol="prediction", \
                 labelCol="medv",metricName="r2")
print("R Squared (R2) on test data = %g" % lr_evaluator.evaluate(lr_predictions))

+------------------+----+--------------------+
|        prediction|medv|            features|
+------------------+----+--------------------+
| 27.67824931238282|22.0|[0.01096,55.0,2.2...|
|30.542503404174163|32.7|[0.01301,35.0,1.5...|
|33.571171050787356|31.6|[0.01432,100.0,1....|
|31.836058974263505|29.1|[0.01439,60.0,2.9...|
|27.833286617331726|24.5|[0.01501,80.0,2.0...|
+------------------+----+--------------------+
only showing top 5 rows
R Squared (R2) on test data = 0.781085


In [18]:
print("numIterations: %d" % trainingSummary.totalIterations)
print("objectiveHistory: %s" % str(trainingSummary.objectiveHistory))
trainingSummary.residuals.show()


numIterations: 0
objectiveHistory: [0.0]
+-------------------+
|          residuals|
+-------------------+
| -6.372784566206999|
| 0.3653625022172733|
|  4.166290115213634|
|  2.925583164258846|
|  9.062054963070175|
|  6.161783775268219|
|  4.400863523113898|
| 1.8597025608312556|
|  9.607454078667406|
|-0.7012995055639912|
|  4.983968963528497|
| -5.870072936876813|
|-4.2227232142175595|
|-0.8146756902700893|
| -4.322769308040332|
|  4.213456583703572|
|  3.293198714215958|
|-0.7995534325292653|
|  2.422358167469593|
| 0.7409011625900348|
+-------------------+
only showing top 20 rows


In [19]:
predictions = lr_model.transform(test_df)
predictions.select("prediction","medv","features").show()

+------------------+----+--------------------+
|        prediction|medv|            features|
+------------------+----+--------------------+
| 27.67824931238282|22.0|[0.01096,55.0,2.2...|
|30.542503404174163|32.7|[0.01301,35.0,1.5...|
|33.571171050787356|31.6|[0.01432,100.0,1....|
|31.836058974263505|29.1|[0.01439,60.0,2.9...|
|27.833286617331726|24.5|[0.01501,80.0,2.0...|
| 44.09003547473384|50.0|[0.01501,90.0,1.2...|
|25.720566208358505|23.1|[0.0187,85.0,4.15...|
|20.329758025133156|20.1|[0.01965,80.0,1.7...|
|43.304317273965566|50.0|[0.02009,95.0,2.6...|
| 32.24580171757717|31.1|[0.02187,60.0,2.9...|
|30.725994901451813|34.7|[0.02729,0.0,7.07...|
|25.326355360320594|21.6|[0.02731,0.0,7.07...|
| 29.89722869413391|24.1|[0.03445,82.5,2.0...|
| 42.16524592569389|48.5|[0.0351,95.0,2.68...|
|38.801624733515766|45.4|[0.03578,20.0,3.3...|
| 32.10313712111193|27.9|[0.03615,80.0,4.9...|
| 34.26615069197753|35.4|[0.03705,20.0,3.3...|
|27.673087604862857|22.0|[0.03932,0.0,3.41...|
| 36.54379452

## Decision tree regression

In [20]:
from pyspark.ml.regression import DecisionTreeRegressor
dt = DecisionTreeRegressor(featuresCol ='features', labelCol = 'medv')
dt_model = dt.fit(train_df)
dt_predictions = dt_model.transform(test_df)
dt_evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="prediction", metricName="rmse")
rmse = dt_evaluator.evaluate(dt_predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

Root Mean Squared Error (RMSE) on test data = 3.79804


In [21]:
from pyspark.ml.regression import DecisionTreeRegressor
dt = DecisionTreeRegressor(featuresCol ='features', labelCol = 'medv')
dt_model = dt.fit(train_df)
dt_predictions = dt_model.transform(test_df)
dt_evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="prediction", metricName="r2")
r2 = dt_evaluator.evaluate(dt_predictions)
print("R2 on test data = %g" % r2)

R2 on test data = 0.824948


In [22]:
lr_evaluator.evaluate(dt_predictions)

0.8249479973883349

In [23]:
 dt_model.featureImportances

SparseVector(13, {0: 0.0007, 2: 0.0, 4: 0.0217, 5: 0.2503, 6: 0.0071, 7: 0.0759, 8: 0.0092, 9: 0.0154, 10: 0.0297, 11: 0.0109, 12: 0.5791})

In [24]:
house_df.take(1)

[Row(_c0=1, crim=0.00632, zn=18.0, indus=2.31, chas=0, nox=0.538, rm=6.575, age=65.2, dis=4.09, rad=1, tax=296, ptratio=15.3, black=396.9, lstat=4.98, medv=24.0)]

## Gradient-boosted tree regression

In [25]:
from pyspark.ml.regression import GBTRegressor
gbt = GBTRegressor(featuresCol = 'features', labelCol = 'medv', maxIter=10)
gbt_model = gbt.fit(train_df)
gbt_predictions = gbt_model.transform(test_df)
gbt_predictions.select('prediction', 'medv', 'features').show(5)

+------------------+----+--------------------+
|        prediction|medv|            features|
+------------------+----+--------------------+
|23.862557614487148|22.0|[0.01096,55.0,2.2...|
| 33.80085250517271|32.7|[0.01301,35.0,1.5...|
|24.648369208009303|31.6|[0.01432,100.0,1....|
|24.401232129683393|29.1|[0.01439,60.0,2.9...|
| 25.32850117163208|24.5|[0.01501,80.0,2.0...|
+------------------+----+--------------------+
only showing top 5 rows


In [26]:
gbt_evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="prediction", metricName="rmse")
rmse = gbt_evaluator.evaluate(gbt_predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

Root Mean Squared Error (RMSE) on test data = 3.42047


In [27]:
gbt_evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="prediction", metricName="r2")
r2 = gbt_evaluator.evaluate(gbt_predictions)
print("R2 Score on test data = %g" % r2)

R2 Score on test data = 0.858023
